<a href="https://colab.research.google.com/github/Toqa-Hamdy/linear-models-and-svm/blob/main/Assignment_2_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Tasks 11 and 12: stable logistic regression

def stable_sigmoid(score_values):
    """Calculate sigmoid without overflowing for large positive or negative scores."""
    score_values = np.asarray(score_values, dtype=float)
    probabilities = np.empty_like(score_values)
    positive_mask = score_values >= 0
    negative_mask = ~positive_mask

    probabilities[positive_mask] = 1.0 / (1.0 + np.exp(-score_values[positive_mask]))
    negative_exponentials = np.exp(score_values[negative_mask])
    probabilities[negative_mask] = negative_exponentials / (1.0 + negative_exponentials)
    return probabilities


def logistic_data_loss(weights, features_with_bias, true_labels):
    score_values = features_with_bias @ weights
    individual_losses = np.logaddexp(0.0, score_values) - true_labels * score_values
    return np.mean(individual_losses)


def logistic_objective(weights, features_with_bias, true_labels, regularization_strength):
    return (
        logistic_data_loss(weights, features_with_bias, true_labels)
        + 0.5 * regularization_strength * np.sum(weights[1:] ** 2)
    )


def logistic_gradient(weights, features_with_bias, true_labels, regularization_strength):
    number_of_samples = len(true_labels)
    score_values = features_with_bias @ weights
    predicted_probabilities = stable_sigmoid(score_values)
    gradient = features_with_bias.T @ (predicted_probabilities - true_labels) / number_of_samples
    gradient[1:] += regularization_strength * weights[1:]
    return gradient


def train_logistic_batch(training_features, training_labels, validation_features,
                         validation_labels, learning_rate, number_of_epochs,
                         regularization_strength):
    training_features_with_bias = add_bias_column(training_features)
    validation_features_with_bias = add_bias_column(validation_features)
    weights = np.zeros(training_features_with_bias.shape[1])

    training_loss_history = []
    validation_loss_history = []
    training_accuracy_history = []
    validation_accuracy_history = []

    for epoch_number in range(number_of_epochs):
        gradient = logistic_gradient(
            weights, training_features_with_bias, training_labels, regularization_strength
        )
        weights -= learning_rate * gradient

        training_probabilities = stable_sigmoid(training_features_with_bias @ weights)
        validation_probabilities = stable_sigmoid(validation_features_with_bias @ weights)
        training_predictions = (training_probabilities >= 0.5).astype(int)
        validation_predictions = (validation_probabilities >= 0.5).astype(int)

        training_loss_history.append(
            logistic_data_loss(weights, training_features_with_bias, training_labels)
        )
        validation_loss_history.append(
            logistic_data_loss(weights, validation_features_with_bias, validation_labels)
        )
        training_accuracy_history.append(
            classification_accuracy(training_labels, training_predictions)
        )
        validation_accuracy_history.append(
            classification_accuracy(validation_labels, validation_predictions)
        )

        if not np.all(np.isfinite(weights)):
            break

    return weights, {
        "training_loss": np.asarray(training_loss_history),
        "validation_loss": np.asarray(validation_loss_history),
        "training_accuracy": np.asarray(training_accuracy_history),
        "validation_accuracy": np.asarray(validation_accuracy_history)
    }


# Numerical-stability check from Section 3.3.
large_scores = np.array([800.0, 800.0, -800.0, -800.0])
large_labels = np.array([1.0, 0.0, 1.0, 0.0])
print("Scores:", large_scores, " labels:", large_labels)
print("Stable sigmoid:", stable_sigmoid(large_scores))
print("Stable per-sample losses (expected 0, 800, 800, 0):",
      np.logaddexp(0.0, large_scores) - large_labels * large_scores)
with np.errstate(over="ignore", divide="ignore", invalid="ignore"):
    naive_probabilities = 1.0 / (1.0 + np.exp(-large_scores))
    naive_losses = -(
        large_labels * np.log(naive_probabilities)
        + (1.0 - large_labels) * np.log(1.0 - naive_probabilities)
    )
print("Naive sigmoid:", naive_probabilities)
print("Naive losses (may be nan / inf):", naive_losses)

cancer_training_targets = breast_cancer_splits[3]
cancer_validation_targets = breast_cancer_splits[4]
cancer_test_targets = breast_cancer_splits[5]
cancer_training_with_bias = add_bias_column(cancer_train)

# Required gradient check with lambda = 0.1.
logistic_gradient_check_generator = np.random.default_rng(42)
random_logistic_weights = logistic_gradient_check_generator.normal(
    0.0, 0.1, size=cancer_training_with_bias.shape[1]
)
logistic_loss_for_check = lambda candidate_weights: logistic_objective(
    candidate_weights, cancer_training_with_bias, cancer_training_targets, 0.1
)
logistic_gradient_for_check = lambda candidate_weights: logistic_gradient(
    candidate_weights, cancer_training_with_bias, cancer_training_targets, 0.1
)
logistic_gradient_error = gradient_check(
    logistic_loss_for_check, logistic_gradient_for_check,
    random_logistic_weights, epsilon=1e-5
)
print("Logistic gradient relative error:", logistic_gradient_error)
assert logistic_gradient_error < 1e-6

regularization_values_logistic = [0.0, 1e-3, 1e-2, 1e-1, 1.0]
logistic_regularization_results = {}
logistic_regularization_rows = []
for regularization_strength in regularization_values_logistic:
    fitted_weights, training_history = train_logistic_batch(
        cancer_train, cancer_training_targets,
        cancer_validation, cancer_validation_targets,
        learning_rate=0.5, number_of_epochs=3000,
        regularization_strength=regularization_strength
    )
    logistic_regularization_results[regularization_strength] = {
        "weights": fitted_weights,
        "history": training_history
    }
    logistic_regularization_rows.append({
        "lambda": regularization_strength,
        "train_accuracy": training_history["training_accuracy"][-1],
        "validation_accuracy": training_history["validation_accuracy"][-1],
        "validation_loss": training_history["validation_loss"][-1],
        "weight_norm_without_bias": np.linalg.norm(fitted_weights[1:])
    })

logistic_regularization_table = pd.DataFrame(logistic_regularization_rows)
display(logistic_regularization_table)
best_logistic_row = logistic_regularization_table.loc[
    logistic_regularization_table["validation_loss"].idxmin()
]
best_logistic_lambda = float(best_logistic_row["lambda"])
best_logistic_weights = logistic_regularization_results[best_logistic_lambda]["weights"]
best_logistic_test_probabilities = stable_sigmoid(add_bias_column(cancer_test) @ best_logistic_weights)
best_logistic_test_predictions = (best_logistic_test_probabilities >= 0.5).astype(int)
print("Selected lambda by validation loss:", best_logistic_lambda)
print("Test accuracy at selected lambda:",
      classification_accuracy(cancer_test_targets, best_logistic_test_predictions))

best_logistic_history = logistic_regularization_results[best_logistic_lambda]["history"]
figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(best_logistic_history["training_loss"], label="Train")
axes[0].plot(best_logistic_history["validation_loss"], label="Validation")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Data cross-entropy loss")
axes[0].set_title("Logistic regression loss")
axes[0].legend()
axes[1].plot(best_logistic_history["training_accuracy"], label="Train")
axes[1].plot(best_logistic_history["validation_accuracy"], label="Validation")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Logistic regression accuracy")
axes[1].legend()
for axis in axes:
    axis.grid(True)
plt.tight_layout()
plt.show()

learning_rates_logistic = [0.01, 0.1, 1.0, 5.0, 20.0]
learning_rate_rows = []
for learning_rate in learning_rates_logistic:
    rate_weights, rate_history = train_logistic_batch(
        cancer_train, cancer_training_targets,
        cancer_validation, cancer_validation_targets,
        learning_rate=learning_rate, number_of_epochs=1000,
        regularization_strength=0.01
    )
    learning_rate_rows.append({
        "learning_rate": learning_rate,
        "final_training_loss": rate_history["training_loss"][-1],
        "finite_weights": bool(np.all(np.isfinite(rate_weights)))
    })
display(pd.DataFrame(learning_rate_rows))
print("Write-up prompt: use the table and curves to identify under-training and failed convergence.")

In [ ]:
#  Task 13: logistic regression without regularization
separability_weight_norm_history = []
separability_loss_history = []
separability_weights = np.zeros(cancer_training_with_bias.shape[1])
for epoch_number in range(20000):
    separability_gradient = logistic_gradient(
        separability_weights, cancer_training_with_bias, cancer_training_targets, 0.0
    )
    separability_weights -= 0.5 * separability_gradient
    separability_weight_norm_history.append(np.linalg.norm(separability_weights[1:]))
    separability_loss_history.append(
        logistic_data_loss(separability_weights, cancer_training_with_bias, cancer_training_targets)
    )

separability_training_accuracy = np.mean(
    (cancer_training_with_bias @ separability_weights > 0) == cancer_training_targets
)
print("Training accuracy with lambda=0 after 20,000 epochs:", separability_training_accuracy)

figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].plot(np.arange(1, 20001), separability_weight_norm_history, label="||w without bias||")
axes[0].set_xscale("log")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Weight norm without bias")
axes[0].set_title("Weight growth without L2")
axes[1].plot(np.arange(1, 20001), np.maximum(separability_loss_history, 1e-15), label="Training loss")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Training cross-entropy loss")
axes[1].set_title("Training loss without L2")
for axis in axes:
    axis.legend()
    axis.grid(True)
plt.tight_layout()
plt.show()
print("Write-up prompt: use the curves to argue whether the training set is linearly separable and explain why L2 changes the optimization problem.")


In [ ]:
# Task 14: compare with scikit-learn logistic regression
logistic_library_rows = []
logistic_library_models = {}
for regularization_strength in regularization_values_logistic:
    if regularization_strength == 0:
        library_model = LogisticRegression(
            penalty=None, solver="lbfgs", max_iter=100000, tol=1e-12
        )
    else:
        library_model = LogisticRegression(
            C=1.0 / (regularization_strength * len(cancer_training_targets)),
            penalty="l2", solver="lbfgs", max_iter=100000, tol=1e-12
        )
    library_model.fit(cancer_train, cancer_training_targets)
    logistic_library_models[regularization_strength] = library_model
    train_predictions = library_model.predict(cancer_train)
    validation_predictions = library_model.predict(cancer_validation)
    test_predictions = library_model.predict(cancer_test)
    logistic_library_rows.append({
        "lambda": regularization_strength,
        "C": (np.inf if regularization_strength == 0 else
              1.0 / (regularization_strength * len(cancer_training_targets))),
        "train_accuracy": accuracy_score(cancer_training_targets, train_predictions),
        "validation_accuracy": accuracy_score(cancer_validation_targets, validation_predictions),
        "test_accuracy": accuracy_score(cancer_test_targets, test_predictions)
    })
display(pd.DataFrame(logistic_library_rows))

# Required weight and prediction comparison at lambda = 0.01.
lambda_for_library_check = 0.01
scratch_lambda_weights = logistic_regularization_results[lambda_for_library_check]["weights"]
library_lambda_model = logistic_library_models[lambda_for_library_check]
library_lambda_weights = np.concatenate((
    [library_lambda_model.intercept_[0]], library_lambda_model.coef_[0]
))
relative_logistic_weight_error = (
    np.linalg.norm(scratch_lambda_weights - library_lambda_weights)
    / max(np.linalg.norm(library_lambda_weights), 1e-12)
)
scratch_lambda_predictions = (
    stable_sigmoid(add_bias_column(cancer_test) @ scratch_lambda_weights) >= 0.5
).astype(int)
library_lambda_predictions = library_lambda_model.predict(cancer_test)
print("Relative weight error at lambda=0.01:", relative_logistic_weight_error)
print("Test prediction agreement:", np.mean(scratch_lambda_predictions == library_lambda_predictions))

# Sweep C as requested.
logistic_c_rows = []
for regularization_parameter_c in [0.001, 0.01, 0.1, 1, 10, 100, 1000]:
    c_model = LogisticRegression(C=regularization_parameter_c, solver="lbfgs",
                                 max_iter=100000, tol=1e-12)
    c_model.fit(cancer_train, cancer_training_targets)
    logistic_c_rows.append({
        "C": regularization_parameter_c,
        "train_accuracy": c_model.score(cancer_train, cancer_training_targets),
        "validation_accuracy": c_model.score(cancer_validation, cancer_validation_targets),
        "test_accuracy": c_model.score(cancer_test, cancer_test_targets)
    })
display(pd.DataFrame(logistic_c_rows))

# Compare solvers at the same C. liblinear also penalizes its intercept.
solver_comparison_rows = []
comparison_c = 1.0 / (lambda_for_library_check * len(cancer_training_targets))
for solver_name in ["lbfgs", "liblinear", "saga"]:
    start_time = time.perf_counter()
    solver_model = LogisticRegression(
        C=comparison_c, solver=solver_name, max_iter=100000,
        tol=1e-10, random_state=42
    )
    solver_model.fit(cancer_train, cancer_training_targets)
    solver_time = time.perf_counter() - start_time
    solver_weights = np.concatenate((
        [solver_model.intercept_[0]], solver_model.coef_[0]
    ))
    solver_comparison_rows.append({
        "solver": solver_name,
        "iterations": int(np.max(solver_model.n_iter_)),
        "fit_time_seconds": solver_time,
        "relative_weight_distance_from_lbfgs": np.nan,
        "intercept": solver_model.intercept_[0],
        "coefficients": solver_model.coef_[0].copy()
    })
    if solver_name == "lbfgs":
        lbfgs_solver_weights = solver_weights
    else:
        solver_comparison_rows[-1]["relative_weight_distance_from_lbfgs"] = (
            np.linalg.norm(solver_weights - lbfgs_solver_weights)
            / max(np.linalg.norm(lbfgs_solver_weights), 1e-12)
        )
solver_comparison_rows[0]["relative_weight_distance_from_lbfgs"] = 0.0
display(pd.DataFrame(solver_comparison_rows))
print("Write-up prompt: discuss solver differences and how penalizing the intercept can shift the fitted boundary.")



### Task 14 — Interpretation

**Verification at λ = 0.01.** With C = 1/(λN), my weights (including the bias) match scikit-learn's to a relative error of about 5e-7 (required < 1e-4), and all test predictions agree. Both minimize the same objective, so the λ ↔ C conversion is correct.

**λ sweep.** For λ > 0 my validation accuracy matches scikit-learn's. The only small difference is at λ = 0: the training set is separable, so there is no finite minimizer and the two optimizers stop at different points.

**C sweep.** C = 0.001 is clearly under-fit (train accuracy 0.88). For C ≥ 100 training accuracy reaches 1.0 while validation accuracy falls (0.956), which is over-fitting. The best validation accuracy is around C = 0.1.

**Solvers (at C = 1/(0.01·N) ≈ 0.293).** lbfgs and saga give the same coefficients (relative difference about 5e-7), but saga needs far more iterations (about 1000 vs about 30) and much more time. **liblinear also penalizes the intercept**: it treats the bias as a normal weight, so the intercept is pulled toward 0 (−0.288 vs −0.424 for lbfgs) and the other weights shift to compensate (about 7% relative difference). That makes it a slightly different model, not just a different solver.

In [ ]:
#  Task 15: confusion matrix, ROC, AUC, and threshold selection

def roc_curve_from_scratch(true_labels, positive_class_scores):
    """Build ROC points by sorting scores and sweeping distinct score thresholds."""
    sort_order = np.argsort(-positive_class_scores)
    sorted_scores = positive_class_scores[sort_order]
    sorted_labels = true_labels[sort_order]

    cumulative_true_positives = np.cumsum(sorted_labels == 1)
    cumulative_false_positives = np.cumsum(sorted_labels == 0)
    last_index_for_each_score = np.where(np.diff(sorted_scores) != 0)[0]
    threshold_indices = np.r_[last_index_for_each_score, len(sorted_scores) - 1]

    true_positive_counts = cumulative_true_positives[threshold_indices]
    false_positive_counts = cumulative_false_positives[threshold_indices]
    total_positive = np.sum(true_labels == 1)
    total_negative = np.sum(true_labels == 0)

    true_positive_rates = np.r_[0.0, true_positive_counts / total_positive]
    false_positive_rates = np.r_[0.0, false_positive_counts / total_negative]
    thresholds = np.r_[np.inf, sorted_scores[threshold_indices]]
    return false_positive_rates, true_positive_rates, thresholds


def area_under_roc_curve(false_positive_rates, true_positive_rates):
    """Trapezoidal rule written by hand (np.trapz was removed in recent NumPy versions)."""
    return np.sum(
        np.diff(false_positive_rates) * (true_positive_rates[1:] + true_positive_rates[:-1]) / 2
    )


lambda_roc_model = logistic_regularization_results[0.01]["weights"]
validation_probabilities = stable_sigmoid(add_bias_column(cancer_validation) @ lambda_roc_model)
test_probabilities = stable_sigmoid(add_bias_column(cancer_test) @ lambda_roc_model)

threshold_half_predictions = (test_probabilities >= 0.5).astype(int)
threshold_half_confusion = confusion_matrix_from_scratch(
    cancer_test_targets, threshold_half_predictions, 2
)
threshold_half_metrics = per_class_precision_recall_f1(threshold_half_confusion)
print("Test confusion matrix at threshold 0.5 (rows=true, columns=predicted):")
print(threshold_half_confusion)
print("Malignant precision / recall / F1:",
      threshold_half_metrics["precision"][1],
      threshold_half_metrics["recall"][1],
      threshold_half_metrics["f1"][1])

validation_fpr, validation_tpr, validation_thresholds = roc_curve_from_scratch(
    cancer_validation_targets, validation_probabilities
)
manual_validation_auc = area_under_roc_curve(validation_fpr, validation_tpr)
library_validation_auc = roc_auc_score(cancer_validation_targets, validation_probabilities)
print("Manual validation AUC:", manual_validation_auc)
print("sklearn validation AUC:", library_validation_auc)
assert abs(manual_validation_auc - library_validation_auc) < 1e-6

plt.figure(figsize=(6, 5))
plt.plot(validation_fpr, validation_tpr, label=f"Scratch ROC (AUC={manual_validation_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate / recall")
plt.title("Validation ROC curve — logistic regression, lambda=0.01")
plt.legend()
plt.grid(True)
plt.show()

validation_positive_scores = validation_probabilities[cancer_validation_targets == 1]
recall_one_threshold = np.min(validation_positive_scores)
threshold_recall_one_predictions = (test_probabilities >= recall_one_threshold).astype(int)
threshold_recall_one_confusion = confusion_matrix_from_scratch(
    cancer_test_targets, threshold_recall_one_predictions, 2
)
threshold_recall_one_metrics = per_class_precision_recall_f1(threshold_recall_one_confusion)
print("Largest validation threshold with recall=1:", recall_one_threshold)
print("Test confusion matrix at selected threshold:")
print(threshold_recall_one_confusion)
print("Malignant precision / recall at selected threshold:",
      threshold_recall_one_metrics["precision"][1],
      threshold_recall_one_metrics["recall"][1])
print("Write-up prompt: explain false-negative and false-positive costs in screening, and uncertainty from only 43 validation positives.")


In [ ]:
# Task 16: decision boundary in original coordinates
selected_cancer_feature_names = ["mean radius", "mean texture"]
selected_cancer_feature_indices = [
    list(breast_cancer_data.feature_names).index(feature_name)
    for feature_name in selected_cancer_feature_names
]
raw_cancer_two_feature_train = breast_cancer_splits[0][:, selected_cancer_feature_indices]
raw_cancer_two_feature_validation = breast_cancer_splits[1][:, selected_cancer_feature_indices]
raw_cancer_two_feature_test = breast_cancer_splits[2][:, selected_cancer_feature_indices]

two_feature_standardizer = Standardizer()
standardized_cancer_two_feature_train = two_feature_standardizer.fit_transform(raw_cancer_two_feature_train)
standardized_cancer_two_feature_validation = two_feature_standardizer.transform(raw_cancer_two_feature_validation)
standardized_cancer_two_feature_test = two_feature_standardizer.transform(raw_cancer_two_feature_test)

two_feature_weights, two_feature_history = train_logistic_batch(
    standardized_cancer_two_feature_train, cancer_training_targets,
    standardized_cancer_two_feature_validation, cancer_validation_targets,
    learning_rate=0.5, number_of_epochs=5000, regularization_strength=0.0
)
original_coordinate_weights = np.empty_like(two_feature_weights)
original_coordinate_weights[1:] = (
    two_feature_weights[1:] / two_feature_standardizer.feature_standard_deviations
)
original_coordinate_weights[0] = (
    two_feature_weights[0]
    - np.sum(two_feature_weights[1:] * two_feature_standardizer.feature_means
             / two_feature_standardizer.feature_standard_deviations)
)

standardized_scores = add_bias_column(standardized_cancer_two_feature_test) @ two_feature_weights
original_scores = add_bias_column(raw_cancer_two_feature_test) @ original_coordinate_weights
print("Maximum score difference after converting weights:",
      np.max(np.abs(standardized_scores - original_scores)))
assert np.max(np.abs(standardized_scores - original_scores)) < 1e-9

all_raw_cancer_two_features = breast_cancer_features[:, selected_cancer_feature_indices]
all_cancer_labels = breast_cancer_targets
plt.figure(figsize=(7, 5))
for class_value, class_name in [(0, "Benign"), (1, "Malignant")]:
    class_mask = all_cancer_labels == class_value
    plt.scatter(
        all_raw_cancer_two_features[class_mask, 0],
        all_raw_cancer_two_features[class_mask, 1],
        label=class_name, alpha=0.55
    )
first_feature_values = np.linspace(
    np.min(all_raw_cancer_two_features[:, 0]),
    np.max(all_raw_cancer_two_features[:, 0]), 200
)
if abs(original_coordinate_weights[2]) > 1e-12:
    second_feature_values = -(
        original_coordinate_weights[0] + original_coordinate_weights[1] * first_feature_values
    ) / original_coordinate_weights[2]
    plt.plot(first_feature_values, second_feature_values, label="Decision boundary")
plt.xlabel("Mean radius")
plt.ylabel("Mean texture")
plt.title("Logistic decision boundary in original feature coordinates")
plt.legend()
plt.grid(True)
plt.show()



## Part D — Softmax classifier (Tasks 17–22)

Digits are divided by 16 and are not standardized. The classifier uses one score column per class, max-subtracted softmax, and cross-entropy computed through log-sum-exp. Bias weights are excluded from the regularization penalty.


In [ ]:
#  Task 17: stable softmax and gradient

def stable_softmax(score_matrix):
    """Calculate row-wise softmax after subtracting each row's largest score."""
    shifted_scores = score_matrix - np.max(score_matrix, axis=1, keepdims=True)
    exponentials = np.exp(shifted_scores)
    return exponentials / np.sum(exponentials, axis=1, keepdims=True)


def softmax_data_loss(weight_matrix, features_with_bias, true_labels):
    score_matrix = features_with_bias @ weight_matrix
    maximum_scores = np.max(score_matrix, axis=1, keepdims=True)
    shifted_scores = score_matrix - maximum_scores
    log_sum_exp = maximum_scores[:, 0] + np.log(np.sum(np.exp(shifted_scores), axis=1))
    correct_class_scores = score_matrix[np.arange(len(true_labels)), true_labels]
    individual_losses = log_sum_exp - correct_class_scores
    return np.mean(individual_losses)


def softmax_objective(weight_matrix, features_with_bias, true_labels, regularization_strength):
    return (
        softmax_data_loss(weight_matrix, features_with_bias, true_labels)
        + 0.5 * regularization_strength * np.sum(weight_matrix[1:, :] ** 2)
    )


def softmax_gradient(weight_matrix, features_with_bias, true_labels, regularization_strength):
    number_of_samples = len(true_labels)
    score_matrix = features_with_bias @ weight_matrix
    probability_matrix = stable_softmax(score_matrix)
    one_hot_labels = np.zeros_like(probability_matrix)
    one_hot_labels[np.arange(number_of_samples), true_labels] = 1.0
    gradient = features_with_bias.T @ (probability_matrix - one_hot_labels) / number_of_samples
    gradient[1:, :] += regularization_strength * weight_matrix[1:, :]
    return gradient


def multiclass_accuracy(weight_matrix, features, true_labels):
    predicted_labels = np.argmax(add_bias_column(features) @ weight_matrix, axis=1)
    return classification_accuracy(true_labels, predicted_labels)


def train_multiclass_mini_batch(training_features, training_labels, validation_features,
                                validation_labels, number_of_classes, learning_rate,
                                number_of_epochs, batch_size, regularization_strength,
                                loss_function, gradient_function, random_generator=None):
    """Train a multiclass linear model with shuffled mini-batches each epoch."""
    if random_generator is None:
        random_generator = np.random.default_rng(42)

    training_features_with_bias = add_bias_column(training_features)
    validation_features_with_bias = add_bias_column(validation_features)
    weight_matrix = np.zeros((training_features_with_bias.shape[1], number_of_classes))
    training_loss_history = []
    validation_loss_history = []
    training_accuracy_history = []
    validation_accuracy_history = []

    for epoch_number in range(number_of_epochs):
        shuffled_indices = random_generator.permutation(len(training_labels))
        for first_index in range(0, len(training_labels), batch_size):
            last_index = min(first_index + batch_size, len(training_labels))
            batch_indices = shuffled_indices[first_index:last_index]
            batch_features = training_features_with_bias[batch_indices]
            batch_labels = training_labels[batch_indices]
            batch_gradient = gradient_function(
                weight_matrix, batch_features, batch_labels, regularization_strength
            )
            weight_matrix -= learning_rate * batch_gradient

        training_loss_history.append(
            loss_function(weight_matrix, training_features_with_bias,
                          training_labels, regularization_strength=0.0)
        )
        validation_loss_history.append(
            loss_function(weight_matrix, validation_features_with_bias,
                          validation_labels, regularization_strength=0.0)
        )
        training_accuracy_history.append(
            np.mean(np.argmax(training_features_with_bias @ weight_matrix, axis=1) == training_labels)
        )
        validation_accuracy_history.append(
            np.mean(np.argmax(validation_features_with_bias @ weight_matrix, axis=1) == validation_labels)
        )

    return weight_matrix, {
        "training_loss": np.asarray(training_loss_history),
        "validation_loss": np.asarray(validation_loss_history),
        "training_accuracy": np.asarray(training_accuracy_history),
        "validation_accuracy": np.asarray(validation_accuracy_history)
    }


# Softmax is unchanged by adding the same constant to every class score:
# exp(s_k+c) / sum_j exp(s_j+c) = exp(c)exp(s_k) / (exp(c)sum_j exp(s_j)).
large_score_example = np.array([[1000.0, 1001.0, 1002.0]])
print("Stable softmax of [1000, 1001, 1002]:", stable_softmax(large_score_example)[0])

# At zero weights, every class has probability 1/10, so cross-entropy is ln(10).
digits_training_with_bias = add_bias_column(digits_train)
digits_validation_with_bias = add_bias_column(digits_validation)
zero_softmax_weights = np.zeros((digits_training_with_bias.shape[1], 10))
zero_weight_loss = softmax_data_loss(
    zero_softmax_weights, digits_training_with_bias, digits_splits[3]
)
print("Softmax loss at W=0:", zero_weight_loss, " expected:", np.log(10))
assert abs(zero_weight_loss - np.log(10)) < 1e-12

softmax_gradient_check_generator = np.random.default_rng(42)
random_softmax_weights = softmax_gradient_check_generator.normal(
    0.0, 0.1, size=(digits_training_with_bias.shape[1], 10)
)
digits_training_targets = digits_splits[3]
softmax_loss_for_check = lambda candidate_weights: softmax_objective(
    candidate_weights, digits_training_with_bias, digits_training_targets, 0.1
)
softmax_gradient_for_check = lambda candidate_weights: softmax_gradient(
    candidate_weights, digits_training_with_bias, digits_training_targets, 0.1
)
softmax_gradient_error = gradient_check(
    softmax_loss_for_check, softmax_gradient_for_check,
    random_softmax_weights, epsilon=1e-5, number_of_entries=100,
    random_generator=np.random.default_rng(42)
)
print("Softmax gradient relative error on 100 entries:", softmax_gradient_error)
assert softmax_gradient_error < 1e-6


In [ ]:
#  Tasks 18 and 19: model selection and library comparison
softmax_learning_rates = [0.01, 0.1, 0.5, 1.0]
softmax_learning_rate_results = {}
softmax_learning_rate_rows = []
for learning_rate in softmax_learning_rates:
    trained_weights, training_history = train_multiclass_mini_batch(
        digits_train, digits_training_targets,
        digits_validation, digits_splits[4],
        number_of_classes=10, learning_rate=learning_rate,
        number_of_epochs=100, batch_size=64, regularization_strength=0.0,
        loss_function=softmax_objective, gradient_function=softmax_gradient,
        random_generator=np.random.default_rng(42)
    )
    softmax_learning_rate_results[learning_rate] = {
        "weights": trained_weights, "history": training_history
    }
    softmax_learning_rate_rows.append({
        "learning_rate": learning_rate,
        "final_train_loss": training_history["training_loss"][-1],
        "final_validation_loss": training_history["validation_loss"][-1],
        "final_train_accuracy": training_history["training_accuracy"][-1],
        "final_validation_accuracy": training_history["validation_accuracy"][-1]
    })
softmax_learning_rate_table = pd.DataFrame(softmax_learning_rate_rows)
display(softmax_learning_rate_table)
best_softmax_learning_rate = float(
    softmax_learning_rate_table.loc[
        softmax_learning_rate_table["final_validation_loss"].idxmin(), "learning_rate"
    ]
)
print("Selected learning rate by validation loss:", best_softmax_learning_rate)

figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for color_index, learning_rate in enumerate(softmax_learning_rates):
    history = softmax_learning_rate_results[learning_rate]["history"]
    color = f"C{color_index}"
    axes[0].plot(history["training_loss"], color=color, label=f"Train, eta={learning_rate}")
    axes[0].plot(history["validation_loss"], color=color, linestyle="--", label=f"Validation, eta={learning_rate}")
    axes[1].plot(history["training_accuracy"], color=color, label=f"Train, eta={learning_rate}")
    axes[1].plot(history["validation_accuracy"], color=color, linestyle="--", label=f"Validation, eta={learning_rate}")
axes[0].set(xlabel="Epoch", ylabel="Cross-entropy loss", title="Softmax loss for each learning rate", yscale="log")
axes[1].set(xlabel="Epoch", ylabel="Accuracy", title="Softmax accuracy for each learning rate", ylim=(0.7, 1.01))
for axis in axes:
    axis.legend(fontsize=7)
    axis.grid(True)
plt.tight_layout()
plt.show()

softmax_regularization_values = [0.0, 1e-4, 1e-3, 1e-2, 1e-1]
softmax_regularization_results = {}
softmax_regularization_rows = []
for regularization_strength in softmax_regularization_values:
    trained_weights, training_history = train_multiclass_mini_batch(
        digits_train, digits_training_targets,
        digits_validation, digits_splits[4],
        number_of_classes=10, learning_rate=best_softmax_learning_rate,
        number_of_epochs=100, batch_size=64,
        regularization_strength=regularization_strength,
        loss_function=softmax_objective, gradient_function=softmax_gradient,
        random_generator=np.random.default_rng(42)
    )
    softmax_regularization_results[regularization_strength] = {
        "weights": trained_weights, "history": training_history
    }
    softmax_regularization_rows.append({
        "lambda": regularization_strength,
        "validation_accuracy": training_history["validation_accuracy"][-1],
        "validation_loss": training_history["validation_loss"][-1]
    })
softmax_regularization_table = pd.DataFrame(softmax_regularization_rows)
display(softmax_regularization_table)
best_softmax_row = softmax_regularization_table.sort_values(
    ["validation_accuracy", "lambda"], ascending=[False, False]
).iloc[0]
best_softmax_lambda = float(best_softmax_row["lambda"])
best_softmax_weights = softmax_regularization_results[best_softmax_lambda]["weights"]
best_softmax_test_predictions = np.argmax(add_bias_column(digits_test) @ best_softmax_weights, axis=1)
print("Selected lambda (validation accuracy; ties choose larger lambda):", best_softmax_lambda)
print("Digits test accuracy (reported once for this selection):",
      classification_accuracy(digits_splits[5], best_softmax_test_predictions))

# scikit-learn multinomial logistic regression using C = 1/(lambda*N_train).
softmax_library_rows = []
softmax_library_models = {}
for regularization_strength in softmax_regularization_values:
    if regularization_strength == 0:
        library_model = LogisticRegression(
            penalty=None, solver="lbfgs", max_iter=10000
        )
    else:
        library_model = LogisticRegression(
            C=1.0 / (regularization_strength * len(digits_training_targets)),
            solver="lbfgs", max_iter=10000
        )
    library_model.fit(digits_train, digits_training_targets)
    softmax_library_models[regularization_strength] = library_model
    softmax_library_rows.append({
        "lambda": regularization_strength,
        "validation_accuracy": library_model.score(digits_validation, digits_splits[4]),
        "test_accuracy": library_model.score(digits_test, digits_splits[5])
    })
display(pd.DataFrame(softmax_library_rows))

softmax_c_sweep_rows = []
for regularization_parameter_c in [0.1, 1, 10, 100]:
    c_model = LogisticRegression(C=regularization_parameter_c, solver="lbfgs", max_iter=10000)
    c_model.fit(digits_train, digits_training_targets)
    softmax_c_sweep_rows.append({
        "C": regularization_parameter_c,
        "validation_accuracy": c_model.score(digits_validation, digits_splits[4]),
        "test_accuracy": c_model.score(digits_test, digits_splits[5])
    })
display(pd.DataFrame(softmax_c_sweep_rows))

library_best_softmax_model = softmax_library_models[best_softmax_lambda]
library_best_softmax_predictions = library_best_softmax_model.predict(digits_test)
print("Scratch/library test prediction agreement at selected lambda:",
      np.mean(best_softmax_test_predictions == library_best_softmax_predictions))

In [ ]:
# Task 20: digits error analysis
best_softmax_test_confusion = confusion_matrix_from_scratch(
    digits_splits[5], best_softmax_test_predictions, 10
)
plt.figure(figsize=(8, 7))
plt.imshow(best_softmax_test_confusion, interpolation="nearest")
plt.title("Digits test confusion matrix")
plt.xlabel("Predicted class")
plt.ylabel("True class")
plt.xticks(np.arange(10))
plt.yticks(np.arange(10))
plt.colorbar(label="Number of images")
for true_class in range(10):
    for predicted_class in range(10):
        plt.text(predicted_class, true_class,
                 str(best_softmax_test_confusion[true_class, predicted_class]),
                 ha="center", va="center", fontsize=8)
plt.tight_layout()
plt.show()

digit_class_metrics = per_class_precision_recall_f1(best_softmax_test_confusion)
class_metrics_table = pd.DataFrame({
    "digit": np.arange(10),
    "precision": digit_class_metrics["precision"],
    "recall": digit_class_metrics["recall"],
    "f1": digit_class_metrics["f1"]
})
display(class_metrics_table)

confusion_pairs = []
for true_class in range(10):
    for predicted_class in range(10):
        if true_class != predicted_class and best_softmax_test_confusion[true_class, predicted_class] > 0:
            confusion_pairs.append((
                best_softmax_test_confusion[true_class, predicted_class],
                true_class, predicted_class
            ))
confusion_pairs.sort(reverse=True)
print("Three most confused true -> predicted pairs:")
for confusion_count, true_class, predicted_class in confusion_pairs[:3]:
    print(f"{true_class} -> {predicted_class}: {confusion_count}")

misclassified_indices = np.where(best_softmax_test_predictions != digits_splits[5])[0][:6]
figure, axes = plt.subplots(1, len(misclassified_indices), figsize=(12, 2.8))
for axis, test_index in zip(np.atleast_1d(axes), misclassified_indices):
    axis.imshow(digits_test[test_index].reshape(8, 8), cmap="gray_r")
    axis.set_title(f"{digits_splits[5][test_index]} / {best_softmax_test_predictions[test_index]}")
    axis.axis("off")
plt.suptitle("Six misclassified test digits: true / predicted")
plt.tight_layout()
plt.show()

# Each class has 64 pixel weights. Reshape them to 8x8 and show side by side.
figure, axes = plt.subplots(1, 10, figsize=(15, 2.5))
for digit_class in range(10):
    axes[digit_class].imshow(best_softmax_weights[1:, digit_class].reshape(8, 8), cmap="coolwarm")
    axes[digit_class].set_title(f"Digit {digit_class}")
    axes[digit_class].axis("off")
plt.suptitle("Softmax class weight maps")
plt.tight_layout()
plt.show()
print("Write-up prompt: identify difficult digit pairs from the table and explain the limitation of a linear boundary on raw pixels.")


### Task 20 — Interpretation

The hardest digits are **1 and 8** (F1 ≈ 0.84 and 0.88). The most confused pairs are **8 → 1, 1 → 9 and 9 → 4**. In 8×8 images these digits share most of their "on" pixels: a vertical stroke in the middle, with a loop or bar on top that is easy to blur at this resolution.

A linear model on raw pixels learns just **one fixed template per class**; the weight maps look like blurry "average digits", and the score is simply how much the input overlaps that template. It cannot handle shifts, thickness changes or different writing styles, and it cannot express "this stroke *or* that stroke". So digits that light up mostly the same pixels get similar scores and are confused.

In [ ]:
# Task 21: two-class softmax equals logistic regression
binary_softmax_random_generator = np.random.default_rng(42)
random_two_class_weights = binary_softmax_random_generator.normal(
    0.0, 1.0, size=(cancer_test.shape[1] + 1, 2)
)
cancer_test_with_bias_for_equivalence = add_bias_column(cancer_test)
two_class_softmax_probabilities = stable_softmax(
    cancer_test_with_bias_for_equivalence @ random_two_class_weights
)[:, 1]
weight_difference_between_classes = (
    random_two_class_weights[:, 1] - random_two_class_weights[:, 0]
)
two_class_logistic_probabilities = stable_sigmoid(
    cancer_test_with_bias_for_equivalence @ weight_difference_between_classes
)
maximum_probability_difference = np.max(
    np.abs(two_class_softmax_probabilities - two_class_logistic_probabilities)
)
print("Maximum K=2 softmax/logistic probability difference:", maximum_probability_difference)
assert maximum_probability_difference < 1e-12
print("Proof idea: divide numerator and denominator by exp(score_0); the class-1 probability becomes sigmoid(score_1-score_0).")



In [ ]:
# Task 22: Iris decision regions
iris_petal_feature_names = ["petal length (cm)", "petal width (cm)"]
iris_petal_feature_indices = [
    list(iris_data.feature_names).index(feature_name)
    for feature_name in iris_petal_feature_names
]
iris_petal_features = iris_features[:, iris_petal_feature_indices]
iris_petal_splits = make_train_validation_test_split(
    iris_petal_features, iris_targets, is_classification=True
)
iris_petal_training_raw, iris_petal_validation_raw, iris_petal_test_raw = iris_petal_splits[:3]
iris_petal_training_targets, iris_petal_validation_targets, iris_petal_test_targets = iris_petal_splits[3:6]
iris_petal_standardizer = Standardizer()
iris_petal_train = iris_petal_standardizer.fit_transform(iris_petal_training_raw)
iris_petal_validation = iris_petal_standardizer.transform(iris_petal_validation_raw)
iris_petal_test = iris_petal_standardizer.transform(iris_petal_test_raw)


def plot_decision_regions(axis, model_predict_function, training_features, training_labels,
                          plot_title):
    first_feature_minimum = np.min(training_features[:, 0]) - 0.7
    first_feature_maximum = np.max(training_features[:, 0]) + 0.7
    second_feature_minimum = np.min(training_features[:, 1]) - 0.7
    second_feature_maximum = np.max(training_features[:, 1]) + 0.7
    first_feature_grid, second_feature_grid = np.meshgrid(
        np.linspace(first_feature_minimum, first_feature_maximum, 250),
        np.linspace(second_feature_minimum, second_feature_maximum, 250)
    )
    mesh_points = np.column_stack((first_feature_grid.ravel(), second_feature_grid.ravel()))
    mesh_predictions = model_predict_function(mesh_points).reshape(first_feature_grid.shape)
    axis.contourf(first_feature_grid, second_feature_grid, mesh_predictions, alpha=0.25,
                  levels=[-0.5, 0.5, 1.5, 2.5], colors=["C0", "C1", "C2"])
    for class_value, class_name in enumerate(iris_data.target_names):
        class_mask = training_labels == class_value
        axis.scatter(training_features[class_mask, 0], training_features[class_mask, 1],
                     color=f"C{class_value}", edgecolors="black", s=25, label=class_name)
    axis.set_xlabel("Standardized petal length")
    axis.set_ylabel("Standardized petal width")
    axis.set_title(plot_title)
    axis.legend()
    axis.grid(True)

iris_models_by_lambda = {}
for regularization_strength in [0.0, 1.0]:
    iris_weights, _ = train_multiclass_mini_batch(
        iris_petal_train, iris_petal_training_targets,
        iris_petal_validation, iris_petal_validation_targets,
        number_of_classes=3, learning_rate=0.5, number_of_epochs=2000,
        batch_size=len(iris_petal_training_targets),
        regularization_strength=regularization_strength,
        loss_function=softmax_objective, gradient_function=softmax_gradient,
        random_generator=np.random.default_rng(42)
    )
    iris_models_by_lambda[regularization_strength] = iris_weights

figure, axes = plt.subplots(2, 2, figsize=(12, 10))
for row_number, regularization_strength in enumerate([0.0, 1.0]):
    own_weights = iris_models_by_lambda[regularization_strength]
    own_predict = lambda mesh_features, active_weights=own_weights: np.argmax(
        add_bias_column(mesh_features) @ active_weights, axis=1
    )
    plot_decision_regions(
        axes[row_number, 0], own_predict, iris_petal_train,
        iris_petal_training_targets, f"Scratch softmax, lambda={regularization_strength}"
    )

    if regularization_strength == 0.0:
        iris_library_model = LogisticRegression(penalty=None, solver="lbfgs", max_iter=10000)
    else:
        iris_library_model = LogisticRegression(
            C=1.0 / (regularization_strength * len(iris_petal_training_targets)),
            solver="lbfgs", max_iter=10000
        )
    iris_library_model.fit(iris_petal_train, iris_petal_training_targets)
    plot_decision_regions(
        axes[row_number, 1], iris_library_model.predict, iris_petal_train,
        iris_petal_training_targets, f"sklearn logistic regression, lambda={regularization_strength}"
    )
plt.tight_layout()
plt.show()
